In [3]:
import json
import chess
import chess.engine
import pandas as pd

STOCKFISH_PATH = "/home/ali/Downloads/stockfish/stockfish-ubuntu-x86-64-avx2"
HERO_USER = "alireza2003"

raw_data = pd.read_json('Alireza2003_original.jsonl', lines=True)

def classify_move(cp_loss):
    """
    Standard Lichess thresholds:
    > 200: Blunder (??)
    90 - 200: Mistake (?)
    30 - 90: Inaccuracy (?!)
    < 30: Good/Normal
    """
    if cp_loss >= 200:
        return "Blunder"
    elif cp_loss >= 90:
        return "Mistake"
    elif cp_loss >= 30:
        return "Inaccuracy"
    else:
        return "Good"

def analyze_games(games_df):
    try:
        engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
    except FileNotFoundError:
        print("ERROR: Stockfish not found. Check your path!")
        return pd.DataFrame()

    data_rows = []
    games_list = games_df.to_dict('records')

    print(f"Starting analysis of {len(games_list)} games...")

    for idx, game in enumerate(games_list[:20]):
        print(f"Analyzing game {idx+1}/{len(games_list)}...")
        board = chess.Board()
        game_id = game['id']
        
        hero_color = chess.WHITE if game['white'] == HERO_USER else chess.BLACK
        
        moves_list = game['moves'] 
        if isinstance(moves_list, str):
            moves_list = moves_list.split()

        for i, move_san in enumerate(moves_list):
            turn_color = board.turn
            is_hero_turn = (turn_color == hero_color)

            try:
                move_obj = board.parse_san(move_san)
                piece = board.piece_at(move_obj.from_square)
                piece_name = chess.piece_name(piece.piece_type).title() if piece else "Unknown"
            except ValueError:
                print(f"Error parsing move {move_san}")
                break

            # 1. Analyze BEFORE
            info = engine.analyse(board, chess.engine.Limit(depth=15))
            score_before = info["score"].white().score(mate_score=10000)
            
            best_move_obj = info["pv"][0] if "pv" in info else None
            best_move_text = board.san(best_move_obj) if best_move_obj else None
            fen_before = board.fen()
            
            # 2. Make Move
            board.push(move_obj)

            # 3. Analyze AFTER
            info_after = engine.analyse(board, chess.engine.Limit(depth=15))
            score_after = info_after["score"].white().score(mate_score=10000)

            # 4. Calculate Loss
            if turn_color == chess.WHITE:
                cp_loss = score_before - score_after
            else:
                cp_loss = score_after - score_before

            # --- Step B: Categorize the Move ---
            category = classify_move(cp_loss)

            # 5. Save Data
            if is_hero_turn:
                row = {
                    "game_id": game_id,
                    "move_number": i + 1,
                    "move_played": move_san,
                    "piece_type": piece_name,
                    "best_move": best_move_text,
                    "cp_loss": cp_loss,
                    "move_category": category,
                    "fen_before": fen_before,
                    "score_before": score_before,
                    "score_after": score_after
                }
                data_rows.append(row)

    engine.quit()
    return pd.DataFrame(data_rows)

try:
    df = analyze_games(raw_data)
    
    if not df.empty:
        df.to_csv("my_chess_data.csv", index=False)
        print("\nSaved to 'my_chess_data.csv'")
        
        print("\n--- PREVIEW OF CATEGORIES ---")
        print(df[['move_played', 'cp_loss', 'move_category']].head(10))
        
        print("\n--- ERROR SUMMARY ---")
        print(df['move_category'].value_counts())
    else:
        print("No data was processed.")

except Exception as e:
    print(f"An error occurred: {e}")

Starting analysis of 652 games...
Analyzing game 1/652...
Analyzing game 2/652...
Analyzing game 3/652...
Analyzing game 4/652...
Analyzing game 5/652...
Analyzing game 6/652...
Analyzing game 7/652...
Analyzing game 8/652...
Analyzing game 9/652...
Analyzing game 10/652...
Analyzing game 11/652...
Analyzing game 12/652...
Analyzing game 13/652...
Analyzing game 14/652...
Analyzing game 15/652...
Analyzing game 16/652...
Analyzing game 17/652...
Analyzing game 18/652...
Analyzing game 19/652...
Analyzing game 20/652...

Saved to 'my_chess_data.csv'

--- PREVIEW OF CATEGORIES ---
  move_played  cp_loss move_category
0          c6        4          Good
1          d5        4          Good
2         Bf5        5          Good
3          e6       -6          Good
4         Bg6        6          Good
5          c5       17          Good
6        Bxc5        8          Good
7         Bb6        0          Good
8          h5       35    Inaccuracy
9         Ne7        4          Good

--- ER

## Clean Data

In [ ]:
df = pd.read_csv("my_chess_data.csv")
print(f"Total moves analyzed: {len(df)}")

# 1. Fix cp_loss: if player played the best move, set to 0
df.loc[df['move_played'] == df['best_move'], 'cp_loss'] = 0

# 2. Clip any remaining negative values (engine variance)
df['cp_loss'] = df['cp_loss'].clip(lower=0)
df['cp_loss_capped'] = df['cp_loss'].clip(upper=1000)

# 3. Re-classify moves after fixing cp_loss
df['move_category'] = df['cp_loss'].apply(classify_move)

# 4. Remove rows with missing best_move
df = df.dropna(subset=['best_move'])

# 5. Add game phase based on move number
def get_game_phase(move_num):
    if move_num <= 15:
        return "Opening"
    elif move_num <= 40:
        return "Middlegame"
    else:
        return "Endgame"

df['game_phase'] = df['move_number'].apply(get_game_phase)

# 6. Filter to only mistakes (puzzle candidates)
puzzles_df = df[df['move_category'].isin(['Blunder', 'Mistake', 'Inaccuracy'])].copy()

# 7. Save cleaned datasets
df.to_csv("my_chess_data_cleaned.csv", index=False)
puzzles_df.to_csv("puzzle_candidates.csv", index=False)

# ============================================
# SUMMARY STATISTICS
# ============================================
print("\n--- CLEANED DATA SUMMARY ---")
print(f"Total moves after cleaning: {len(df)}")
print(f"\nMove categories:")
print(df['move_category'].value_counts())

print(f"\n--- PUZZLE CANDIDATES ---")
print(f"Total puzzle candidates: {len(puzzles_df)}")
print(f"\nBy category:")
print(puzzles_df['move_category'].value_counts())
print(f"\nBy piece type:")
print(puzzles_df['piece_type'].value_counts())
print(f"\nBy game phase:")
print(puzzles_df['game_phase'].value_counts())

print("\n✅ Saved 'my_chess_data_cleaned.csv' (all moves)")
print("✅ Saved 'puzzle_candidates.csv' (mistakes only)")

Total moves analyzed: 821

--- CLEANED DATA SUMMARY ---
Total moves after cleaning: 821

Move categories:
move_category
Good          646
Inaccuracy    122
Mistake        33
Blunder        20
Name: count, dtype: int64

--- PUZZLE CANDIDATES ---
Total puzzle candidates: 175

By category:
move_category
Inaccuracy    122
Mistake        33
Blunder        20
Name: count, dtype: int64

By piece type:
piece_type
Pawn      45
Queen     30
Rook      30
Bishop    26
Knight    25
King      19
Name: count, dtype: int64

By game phase:
game_phase
Endgame       95
Middlegame    64
Opening       16
Name: count, dtype: int64

✅ Saved 'my_chess_data_cleaned.csv' (all moves)
✅ Saved 'puzzle_candidates.csv' (mistakes only)


# Player Weakness Analysis

In [ ]:
# ============================================
# PLAYER WEAKNESS PATTERN ANALYSIS
# ============================================

# Load puzzle candidates
puzzles = pd.read_csv("puzzle_candidates.csv")

# Cap extreme values (mate-related) for cleaner analysis
puzzles['cp_loss_capped'] = puzzles['cp_loss'].clip(upper=500)

print("=" * 60)
print("🎯 PLAYER WEAKNESS ANALYSIS")
print("=" * 60)

# 1. Mistakes by Piece Type
print("\n📊 MISTAKES BY PIECE TYPE")
print("-" * 40)
piece_analysis = puzzles.groupby('piece_type').agg({
    'cp_loss_capped': ['count', 'mean', 'sum']
}).round(1)
piece_analysis.columns = ['Count', 'Avg CP Loss', 'Total CP Loss']
piece_analysis = piece_analysis.sort_values('Total CP Loss', ascending=False)
print(piece_analysis)

# 2. Mistakes by Game Phase
print("\n📊 MISTAKES BY GAME PHASE")
print("-" * 40)
phase_analysis = puzzles.groupby('game_phase').agg({
    'cp_loss_capped': ['count', 'mean', 'sum']
}).round(1)
phase_analysis.columns = ['Count', 'Avg CP Loss', 'Total CP Loss']
print(phase_analysis)

# 3. Mistakes by Category and Phase
print("\n📊 ERROR SEVERITY BY GAME PHASE")
print("-" * 40)
severity_phase = pd.crosstab(puzzles['game_phase'], puzzles['move_category'])
print(severity_phase)

# 4. Worst Individual Mistakes
print("\n🚨 TOP 10 WORST MISTAKES")
print("-" * 40)
worst = puzzles.nsmallest(10, 'cp_loss') if puzzles['cp_loss'].min() < 0 else puzzles.nlargest(10, 'cp_loss')
worst = puzzles.nlargest(10, 'cp_loss')[['game_id', 'move_number', 'move_played', 'best_move', 'cp_loss', 'piece_type', 'game_phase']]
print(worst.to_string(index=False))

# 5. Key Insights
print("\n" + "=" * 60)
print("💡 KEY INSIGHTS FOR PUZZLE GENERATION")
print("=" * 60)

# Most problematic piece
worst_piece = piece_analysis['Total CP Loss'].idxmax()
worst_piece_count = piece_analysis.loc[worst_piece, 'Count']
print(f"\n⚠️  Most problematic piece: {worst_piece} ({int(worst_piece_count)} mistakes)")

# Most problematic phase
worst_phase = phase_analysis['Total CP Loss'].idxmax()
worst_phase_count = phase_analysis.loc[worst_phase, 'Count']
print(f"⚠️  Most problematic phase: {worst_phase} ({int(worst_phase_count)} mistakes)")

# Blunder count
blunder_count = len(puzzles[puzzles['move_category'] == 'Blunder'])
mistake_count = len(puzzles[puzzles['move_category'] == 'Mistake'])
print(f"\n📈 Blunders: {blunder_count} | Mistakes: {mistake_count} | Inaccuracies: {len(puzzles) - blunder_count - mistake_count}")

# Recommendation
print("\n🎯 PUZZLE RECOMMENDATIONS:")
print(f"   → Focus on {worst_piece} tactics")
print(f"   → Practice {worst_phase} positions")
if blunder_count > 5:
    print(f"   → Prioritize blunder prevention training")